## Install Dependencies

In [1]:
%%capture
!pip install evaluate rouge_score nltk sacrebleu
!pip uninstall -y unsloth unsloth-zoo peft trl transformers
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps packaging ninja einops flash-attn xformers trl peft accelerate bitsandbytes unsloth-zoo

## Import Libraries and Setup

In [2]:
import torch
from unsloth import FastLanguageModel
import json
import pandas as pd
from datasets import Dataset
import os

max_seq_length = 2048 
dtype = None 
load_in_4bit = True 

print(f"GPU Model: {torch.cuda.get_device_name(0)}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-04-10 09:50:16.736996: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775814616.948665      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775814617.037743      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775814617.912982      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775814617.913018      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775814617.913021      23 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!
GPU Model: Tesla T4


## Load, Merge, and Count Data

In [3]:
train_file_paths = [
    "/kaggle/input/nusantara-law-corpus/Adagium/adagium-all-reformat.json",
    "/kaggle/input/nusantara-law-corpus/GBHN/GBHN-all-reformat.json",
    "/kaggle/input/nusantara-law-corpus/Glosarium-MA/GMA-all.json",
    "/kaggle/input/nusantara-law-corpus/HukumOnline/HO-all.json",
    "/kaggle/input/nusantara-law-corpus/KHPTSultra/KHPTS-all.json",
    "/kaggle/input/nusantara-law-corpus/LawDictionary/LD-all.json",
    "/kaggle/input/nusantara-law-corpus/TAP-MPR/TMPR-all-reformat.json",
    "/kaggle/input/nusantara-law-corpus/UUD/uud-id-reformat.json"
]

test_file_path = "/kaggle/input/nusantara-law-corpus/test-data-reformat.json"

combined_train_data = []

for file_path in train_file_paths:
    if os.path.exists(file_path):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                if isinstance(data, list):
                    combined_train_data.extend(data)
                    print(f"Successfully loaded {len(data)} records from: {os.path.basename(file_path)}")
                else:
                    print(f"Warning: {file_path} format is not a list of records.")
        except Exception as e:
            print(f"Error reading {file_path}: {e}")
    else:
        print(f"File not found: {file_path}")

test_data = []
if os.path.exists(test_file_path):
    try:
        with open(test_file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            if isinstance(data, list):
                test_data.extend(data)
                print(f"Successfully loaded {len(data)} test records from: {os.path.basename(test_file_path)}")
    except Exception as e:
        print(f"Error reading {test_file_path}: {e}")
else:
    print(f"Test file not found: {test_file_path}")

train_df = pd.DataFrame(combined_train_data)
test_df = pd.DataFrame(test_data)

print(f"Total train data points: {len(train_df)}")
print(f"Total test data points: {len(test_df)}")

raw_train_dataset = Dataset.from_pandas(train_df)
raw_eval_dataset = Dataset.from_pandas(test_df)

Successfully loaded 89 records from: adagium-all-reformat.json
Successfully loaded 106 records from: GBHN-all-reformat.json
Successfully loaded 207 records from: GMA-all.json
Successfully loaded 2342 records from: HO-all.json
Successfully loaded 144 records from: KHPTS-all.json
Successfully loaded 2456 records from: LD-all.json
Successfully loaded 1469 records from: TMPR-all-reformat.json
Successfully loaded 256 records from: uud-id-reformat.json
Successfully loaded 444 test records from: test-data-reformat.json
Total train data points: 7069
Total test data points: 444


## Load Model (Qwen2.5-instruct 7b)

In [4]:
model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"

print(f"Loading Model: {model_name}...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token 

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    contexts     = examples["context"]
    responses    = examples["response"]
    texts = []
    for instruction, context, response in zip(instructions, contexts, responses):
        text = alpaca_prompt.format(instruction, context, response) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts }

train_dataset = raw_train_dataset.map(formatting_prompts_func, batched = True)
eval_dataset = raw_eval_dataset.map(formatting_prompts_func, batched = True)

print("Success! Loaded and formatted data.")

Loading Model: unsloth/Qwen2.5-7B-Instruct-bnb-4bit...
==((====))==  Unsloth 2026.4.4: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

unsloth/Qwen2.5-7B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Map:   0%|          | 0/7069 [00:00<?, ? examples/s]

Map:   0%|          | 0/444 [00:00<?, ? examples/s]

Success! Loaded and formatted data.


## Configure QLoRA Adapters

In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 64, 
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"], 
    lora_alpha = 64, 
    lora_dropout = 0.05, 
    bias = "none",    
    use_gradient_checkpointing = "unsloth", 
    random_state = 3407,
    use_rslora = True, 
    loftq_config = None,
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.4.4 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


## Training (SFTTrainer)

In [6]:
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

sft_config = SFTConfig(
    output_dir="outputs",
    max_seq_length=max_seq_length,
    dataset_text_field="text",
    dataset_num_proc=2,
    packing=False,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8, 
    num_train_epochs=6,    
    learning_rate=2e-4,    
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,      
    optim="adamw_8bit",
    weight_decay=0.01,     
    fp16=True,
    bf16=False,
    eval_strategy="steps",
    eval_steps=175,        
    save_steps=175,        
    logging_steps=10,
    save_total_limit=2,          
    load_best_model_at_end=True, 
    metric_for_best_model="eval_loss", 
    greater_is_better=False,
    report_to="none",
    seed=3407,
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,   
    args = sft_config,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)] 
)

def predict_training_time(dataset, config, seconds_per_step=1.2):
    total_examples = len(dataset)
    batch_size = config.per_device_train_batch_size
    grad_accum = config.gradient_accumulation_steps
    epochs = config.num_train_epochs
    
    steps_per_epoch = total_examples // (batch_size * grad_accum)
    total_steps = steps_per_epoch * epochs
    
    estimated_seconds = total_steps * seconds_per_step
    hours = estimated_seconds // 3600
    minutes = (estimated_seconds % 3600) // 60
    
    print("Prediksi Estimasi Waktu Pelatihan")
    print(f"Total Data Latih: {total_examples} sampel")
    print(f"Total Steps: {total_steps}")
    print(f"Estimasi Waktu: ~{int(hours)} jam dan {int(minutes)} menit")
    print(f"(Berdasarkan asumsi kecepatan ~{seconds_per_step} detik per step pada Tesla T4)")

predict_training_time(train_dataset, sft_config)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/7069 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/444 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Prediksi Estimasi Waktu Pelatihan
Total Data Latih: 7069 sampel
Total Steps: 2646
Estimasi Waktu: ~0 jam dan 52 menit
(Berdasarkan asumsi kecepatan ~1.2 detik per step pada Tesla T4)


## Execute Training

In [7]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,069 | Num Epochs = 6 | Total steps = 2,652
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 161,480,704 of 7,777,097,216 (2.08% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
175,0.971000,1.290918
350,0.941100,1.435923
525,0.701500,1.468515
700,0.710500,1.377771


## Inference and Evaluation

In [8]:
import time
import evaluate
import numpy as np
import nltk
from tqdm import tqdm
import random

# Download NLTK data required for ROUGE and METEOR
nltk.download("punkt")
nltk.download("wordnet")

# Load the evaluation metrics
bleu_metric = evaluate.load("sacrebleu")
rouge_metric = evaluate.load("rouge")
meteor_metric = evaluate.load("meteor")

# Enable native Unsloth inference speeds
FastLanguageModel.for_inference(model)

print("Memulai Evaluasi Metrik secara Komprehensif...")
print("MENGGUNAKAN TRAIN DATASET SEBAGAI TEST DATA (MEMORIZATION TEST)")

# Menggunakan TRAIN_DATASET sesuai permintaan Anda untuk memaksimalkan akurasi
sample_size = min(50, len(train_dataset))
sample_indices = random.sample(range(len(train_dataset)), sample_size)

predictions = []
references = []

# Gunakan prompt yang sama persis
eval_prompt = """Anda adalah seorang pakar hukum Indonesia dan kamus hukum yang sangat presisi. 
Tugas Anda adalah memberikan definisi atau penjelasan hukum yang formal, baku, dan sesuai dengan literatur perundang-undangan. 
Jangan merangkum dengan bahasa santai. Gunakan gaya bahasa hukum yang kaku dan tepat.

### Instruction:
{}

### Input:
{}

### Response:
"""

for idx in tqdm(sample_indices, desc="Generating Responses"):
    sample = train_dataset[idx]
    test_instruction = sample["instruction"]
    test_context = sample.get("context", "")
    target_response = sample["response"]
    
    # Prepare the prompt
    inputs = tokenizer(
        [eval_prompt.format(test_instruction, test_context)],
        return_tensors="pt"
    ).to("cuda")
    
    outputs = model.generate(
        **inputs, 
        max_new_tokens=256, 
        use_cache=True,
        do_sample=False,           
        repetition_penalty=1.15,   
        no_repeat_ngram_size=3     
    )
    
    # Decode
    decoded_output = tokenizer.batch_decode(outputs)[0]
    
    # Ekstrak jawaban setelah "### Response:\n"
    if "### Response:\n" in decoded_output:
        generated_response = decoded_output.split("### Response:\n")[-1].replace(tokenizer.eos_token, "").strip()
    else:
        generated_response = decoded_output.replace(tokenizer.eos_token, "").strip()
        
    predictions.append(generated_response)
    references.append(target_response)

# CALCULATE METRICS
print("\nMenghitung Metrik (BLEU, ROUGE, METEOR)...")

# ROUGE requires newline-separated sentences
preds_rouge = ["\n".join(nltk.sent_tokenize(pred)) for pred in predictions]
refs_rouge = ["\n".join(nltk.sent_tokenize(ref)) for ref in references]

rouge_result = rouge_metric.compute(predictions=preds_rouge, references=refs_rouge, use_stemmer=True)
# SacreBLEU expects references as a list of lists
bleu_result = bleu_metric.compute(predictions=predictions, references=[[ref] for ref in references])
meteor_result = meteor_metric.compute(predictions=predictions, references=references)

print("\n" + "="*40)
print("HASIL EVALUASI MODEL PADA TRAIN DATA")
print("="*40)
print(f"SacreBLEU Score : {bleu_result['score']:.2f}")
print(f"ROUGE-1 Score   : {rouge_result['rouge1']*100:.2f}")
print(f"ROUGE-2 Score   : {rouge_result['rouge2']*100:.2f}")
print(f"ROUGE-L Score   : {rouge_result['rougeL']*100:.2f}")
print(f"METEOR Score    : {meteor_result['meteor']*100:.2f}")
print("="*40)

# Print a few examples to visually inspect the quality
print("\n--- Contoh Perbandingan (Prediction vs Ground Truth)")
for i in range(3):
    print(f"\nContoh {i+1}:")
    print(f"Target : {references[i]}")
    print(f"Prediksi: {predictions[i]}")

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...


Memulai Evaluasi Metrik secara Komprehensif...
MENGGUNAKAN TRAIN DATASET SEBAGAI TEST DATA (MEMORIZATION TEST)


Generating Responses: 100%|██████████| 50/50 [11:26<00:00, 13.73s/it]



Menghitung Metrik (BLEU, ROUGE, METEOR)...

HASIL EVALUASI MODEL PADA TRAIN DATA
SacreBLEU Score : 1.00
ROUGE-1 Score   : 16.24
ROUGE-2 Score   : 4.23
ROUGE-L Score   : 12.83
METEOR Score    : 20.14

--- Contoh Perbandingan (Prediction vs Ground Truth)

Contoh 1:
Target : Anak Korban dalam kacamata sistem peradilan anak adalah identifikasi pada setiap individu anak yang dari segi usianya belum menginjak atau belum genap porsi pada wujud usianya ini belum berumur belum pada usia genap belum wujud dari porsi usianya pada porsi usianya di umurnya ini belum pada wujud belum belum porsi pada belum berumur belum di belum wujud berumur porsi dari usianya ini berumur belum porsi usianya di porsi berumur 18 dari wujud porsi pada usianya ini 18 di porsi wujud 18 usianya porsi 18 di pada 18 di umur 18 pada usia porsi 18 porsi wujud tahun porsi wujud di pada usia 18 wujud porsi 18 porsi pada wujud tahun di tahun porsi 18 di tahun pada wujud tahun porsi usianya porsi tahun di tahun wujud yang pada

## Cleanup Cell

In [9]:
import shutil
import os
import gc
import torch

# Hapus variabel yang tidak terpakai dari RAM
del trainer
gc.collect()

# Bersihkan Cache GPU
torch.cuda.empty_cache()

# Hapus folder checkpoint pelatihan
path_to_clean = "/kaggle/working/outputs"
if os.path.exists(path_to_clean):
    print(f"Cleaning up {path_to_clean} to prevent 'No space left on device' error...")
    try:
        shutil.rmtree(path_to_clean)
        print("Cleanup successful. Disk space reclaimed.")
    except Exception as e:
        print(f"Could not fully clean directory: {e}")
else:
    print(f"Directory {path_to_clean} not found, skipping cleanup.")

!df -h /kaggle/working

Cleaning up /kaggle/working/outputs to prevent 'No space left on device' error...
Cleanup successful. Disk space reclaimed.
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G   19M   20G   1% /kaggle/working


## Save the Model

In [10]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("hf_token")
login(hf_token)

repo_name = "bayhaqieee/qwen2.5-7b-nlaw-gguf"

# PUSH ADAPTERS
print("Pushing Adapters (LoRA) to Hugging Face...")
try:
    model.push_to_hub(repo_name, token=hf_token)
    tokenizer.push_to_hub(repo_name, token=hf_token)
    print("Adapters (LoRA) Pushed to Hugging Face successfully!")
except Exception as e:
    print(f"Adapter Push Failed: {e}")

# PUSH GGUF KE HUGGING FACE
print("\nPushing GGUF to Hugging Face (This requires heavy disk space)...")
try:
    model.push_to_hub_gguf(
        repo_name, 
        tokenizer, 
        quantization_method = "q4_k_m",
        token = hf_token
    )
    print("GGUF Pushed to Hugging Face successfully!")
except Exception as e:
    print(f"\nGGUF Push Failed: {e}")
    print("\nNOTE: Kaggle's 20GB disk limit often blocks 7B GGUF conversions.")
    print("Adapter LoRA sudah berhasil disimpan ke Hugging Face di Langkah 1!")
    print("Gabungkan (merge) LoRA ke Base Model menjadi GGUF secara terpisah di Google Colab.")

Pushing Adapters (LoRA) to Hugging Face...


README.md:   0%|          | 0.00/741 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/bayhaqieee/qwen2.5-7b-nlaw-gguf


...

Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/tmp/unsloth_gguf_knt4_spa_gguf/Qwen2.5-7B-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['/tmp/unsloth_gguf_knt4_spa_gguf/Qwen2.5-7B-Instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model /tmp/unsloth_gguf_knt4_spa_gguf/Qwen2.5-7B-Instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to /tmp/unsloth_gguf_knt4_spa_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f /tmp/unsloth_gguf_knt4_spa_gguf/Modelfile
Unsloth: Uploading GGUF to Huggingface Hub...
Uploading Qwen2.5-7B-Instruct.Q4_K_M.gguf...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.


Uploading config.json...
Uploading Ollama Modelfile...
Unsloth: Successfully uploaded GGUF to https://huggingface.co/bayhaqieee/qwen2.5-7b-nlaw-gguf
Unsloth: Cleaning up temporary files...
GGUF Pushed to Hugging Face successfully!


In [11]:
# print("\nSaving Adapters Locally (Kaggle)")
# local_folder = "qwen2-7b-nlaw_adapter"
# model.save_pretrained(local_folder)
# tokenizer.save_pretrained(local_folder)
# print(f"Adapters saved locally to folder: {local_folder}")